In [2]:
import pandas as pd
import numpy as np
import os
import re
import subprocess
import nltk
import wget
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
from string import punctuation

import pymorphy2
from pymorphy2 import MorphAnalyzer
from gensim.models import Word2Vec, FastText
import gensim.matutils
from gensim.models import KeyedVectors

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import json

In [3]:
# Загрузка данных NLTK
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mrkaz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mrkaz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mrkaz\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [29]:
BASE_PATH = "C:/work projects/ITMO/ZOS/NLP/DATA"

In [5]:
# Загрузка данных
positive_file = BASE_PATH + "/negative.csv"
negative_file = BASE_PATH + "/negative.csv"

# Чтение данных
pos = pd.read_csv(
    positive_file,
    sep=";",
    names=[
        "id",
        "date",
        "name",
        "text",
        "type",
        "rep",
        "rtw",
        "faw",
        "stcount",
        "foll",
        "frien",
        "listcount",
    ],
)
neg = pd.read_csv(
    negative_file,
    sep=";",
    names=[
        "id",
        "date",
        "name",
        "text",
        "type",
        "rep",
        "rtw",
        "faw",
        "stcount",
        "foll",
        "frien",
        "listcount",
    ],
)

# Объединение данных
neg["type"] = neg["type"].replace(-1, 0)
df = pd.concat([pos, neg])

# Удаление ненужных столбцов
df = df.drop(
    [
        "id",
        "date",
        "name",
        "rep",
        "rtw",
        "faw",
        "stcount",
        "foll",
        "frien",
        "listcount",
    ],
    axis=1,
)
dfudpipe = df

In [6]:
df

,text,type
0,на работе был полный пиддес :| и так каждое за...,-1
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1
2,@elina_4post как говорят обещаного три года жд...,-1
3,"Желаю хорошего полёта и удачной посадки,я буду...",-1
4,"Обновил за каким-то лешим surf, теперь не рабо...",-1
...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,0
111919,скучаю так :-( только @taaannyaaa вправляет мо...,0
111920,"Вот и в школу, в говно это идти уже надо(",0
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0


In [7]:
stop_words = set(stopwords.words("russian"))

# Инициализация морфологического анализатора
morph = MorphAnalyzer()


# Функция для нормализации текста
def normalize_text(text):
    # Приведение к нижнему регистру
    text = text.lower()

    # Удаление всех символов, кроме кириллицы и пробелов
    text = re.sub("[^а-яё ]", "", text)

    # Удаление лишних пробелов
    text = re.sub("\\s+", " ", text).strip()

    # Удаление стоп-слов
    words = [word for word in text.split() if word not in stop_words]

    # Лемматизация
    words = [morph.parse(word)[0].normal_form for word in words]

    return " ".join(words)


# Применение нормализации к колонке 'text'
df["normalized_text"] = df["text"].apply(normalize_text)

# Токенизация нормализованного текста
df["tokens"] = df["normalized_text"].apply(word_tokenize)

In [8]:
df

,text,type,normalized_text,tokens
0,на работе был полный пиддес :| и так каждое за...,-1,работа полный пиддес каждый закрытие месяц сви...,"[работа, полный, пиддес, каждый, закрытие, мес..."
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,коллега сидеть рубиться изз долбать винд мочь,"[коллега, сидеть, рубиться, изз, долбать, винд..."
2,@elina_4post как говорят обещаного три года жд...,-1,говорить обещаной год ждать,"[говорить, обещаной, год, ждать]"
3,"Желаю хорошего полёта и удачной посадки,я буду...",-1,желать хороший полёт удачный посадкия быть оче...,"[желать, хороший, полёт, удачный, посадкия, бы..."
4,"Обновил за каким-то лешим surf, теперь не рабо...",-1,обновить какимтый леший работать простоплеер,"[обновить, какимтый, леший, работать, простопл..."
...,...,...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[каждый, хотеть, исправлять]"
111919,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать, вправлять, мозг, равно, скучать]"
111920,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[школа, говно, это, идти]"
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[тауриэль, грусть, обнять]"


In [9]:
# Загрузка тонального словаря
tonality_dict = pd.read_csv(
    BASE_PATH + "/words_all_full_rating.csv", encoding="windows-1251", delimiter=";"
)

In [10]:
tonality_dict

,Words,mean,dispersion,average rate,Unnamed: 4
0,аборигенный,"-0,25","0,433012701892219",0,NaN
1,аборт,-1,"0,816496580927726",-1,NaN
2,абрамович,0,0,0,NaN
3,абсолютный,"0,333333333333333","0,471404520791032",0,NaN
4,абстрактный,"-0,111111111111111","0,87488976377909",0,NaN
...,...,...,...,...,...
7540,ярый,"-0,333333333333333","0,942809041582063",0,NaN
7541,ясно,0,0,0,NaN
7542,ясность,"0,666666666666667","0,471404520791032",1,NaN
7543,ясный,"0,666666666666667","0,471404520791032",1,NaN


In [11]:
tonality_dict["mean"] = tonality_dict["mean"].str.replace(",", ".").astype(float)
tonality_dict = dict(zip(tonality_dict["Words"], tonality_dict["mean"]))

In [12]:
tonality_dict

{'аборигенный': -0.25,
 'аборт': -1.0,
 'абрамович': 0.0,
 'абсолютный': 0.333333333333333,
 'абстрактный': -0.111111111111111,
 'абсурд': -1.0,
 'абсурдно': -1.0,
 'абсурдный': -1.0,
 'абхаз': 0.0,
 'абхазия': 0.0,
 'авантюра': -0.333333333333333,
 'авантюрный': -0.333333333333333,
 'аварийный': -1.0,
 'авария': -1.25,
 'август': 0.0,
 'авианосец': -0.333333333333333,
 'авиационный': 0.0,
 'авиация': 0.111111111111111,
 'австралиец': -0.333333333333333,
 'австралийский': 0.0,
 'австралия': 0.0,
 'австриец': 0.0,
 'австрийский': 0.0,
 'австрия': 0.0,
 'австро-венгрия': 0.0,
 'авто': 0.0,
 'автобиографический': 0.0,
 'автовладелец': 0.0,
 'автодорога': 0.0,
 'автолюбитель': 0.222222222222222,
 'автомат': -0.666666666666667,
 'автоматический': -0.222222222222222,
 'автоматчик': 0.0,
 'автомобилист': 0.0,
 'автомобильный': 0.0,
 'автономный': 0.0,
 'автор': 0.0,
 'автореферат': 0.0,
 'авторитет': 0.333333333333333,
 'авторитетный': 0.666666666666667,
 'автотранспорт': 0.0,
 'автошкола': 0

In [13]:
# Функция для извлечения тональных признаков
def extract_tonality_features(tokens):
    # Получаем тональные значения для токенов, которые есть в словаре
    tonality_scores = [
        tonality_dict.get(token, 0) for token in tokens if token in tonality_dict
    ]

    # Если список пуст, возвращаем [0, 0, 0, 0, 0, 0]
    if not tonality_scores:
        return [0, 0, 0, 0, 0, 0]

    # Вычисляем признаки
    mean_tonality = sum(tonality_scores) / len(tonality_scores)  # Среднее значение
    max_tonality = max(tonality_scores)  # Максимальное значение
    min_tonality = min(tonality_scores)  # Минимальное значение
    sum_tonality = sum(tonality_scores)  # Суммарное значение
    pos_count = len(
        [score for score in tonality_scores if score > 0]
    )  # Количество положительных значений
    neg_count = len(
        [score for score in tonality_scores if score < 0]
    )  # Количество отрицательных значений

    return [
        mean_tonality,
        max_tonality,
        min_tonality,
        sum_tonality,
        pos_count,
        neg_count,
    ]


# Применение функции к токенам
df["tonality_features"] = df["tokens"].apply(extract_tonality_features)

# Разделение списка признаков на отдельные колонки
df[
    [
        "tonality_mean",
        "tonality_max",
        "tonality_min",
        "tonality_sum",
        "tonality_pos_count",
        "tonality_neg_count",
    ]
] = pd.DataFrame(df["tonality_features"].tolist(), index=df.index)

# Удаление временной колонки с признаками
df.drop(columns=["tonality_features"], inplace=True)

# Сохранение результатов в файл
df.to_csv("text_with_tonality_features.csv", index=False)

In [14]:
df

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count
0,на работе был полный пиддес :| и так каждое за...,-1,работа полный пиддес каждый закрытие месяц сви...,"[работа, полный, пиддес, каждый, закрытие, мес...",0.129630,0.222222,0.000000,0.388889,2,0
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,коллега сидеть рубиться изз долбать винд мочь,"[коллега, сидеть, рубиться, изз, долбать, винд...",-0.200000,-0.200000,-0.200000,-0.200000,0,1
2,@elina_4post как говорят обещаного три года жд...,-1,говорить обещаной год ждать,"[говорить, обещаной, год, ждать]",0.000000,0.000000,0.000000,0.000000,0,0
3,"Желаю хорошего полёта и удачной посадки,я буду...",-1,желать хороший полёт удачный посадкия быть оче...,"[желать, хороший, полёт, удачный, посадкия, бы...",1.111111,1.222222,1.000000,2.222222,2,0
4,"Обновил за каким-то лешим surf, теперь не рабо...",-1,обновить какимтый леший работать простоплеер,"[обновить, какимтый, леший, работать, простопл...",0.000000,0.000000,0.000000,0.000000,0,0
...,...,...,...,...,...,...,...,...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[каждый, хотеть, исправлять]",0.000000,0.000000,0.000000,0.000000,0,0
111919,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать, вправлять, мозг, равно, скучать]",0.000000,0.000000,0.000000,0.000000,0,0
111920,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[школа, говно, это, идти]",0.000000,0.000000,0.000000,0.000000,0,0
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[тауриэль, грусть, обнять]",0.000000,0.000000,0.000000,0.000000,0,0


In [15]:
# Инициализация морфологического анализатора
morph = MorphAnalyzer()


# Функция для извлечения морфологических признаков
def extract_morph_features(tokens):
    # Словарь для подсчета частей речи
    pos_counts = {"NOUN": 0, "VERB": 0, "ADJ": 0, "ADV": 0, "OTHER": 0}

    # Подсчет частей речи
    for token in tokens:
        parsed_word = morph.parse(token)[0]  # Анализ токена
        pos = parsed_word.tag.POS  # Получение части речи

        # Увеличиваем счетчик для соответствующей части речи
        if pos in pos_counts:
            pos_counts[pos] += 1
        else:
            pos_counts["OTHER"] += 1

    # Вычисление относительной частоты
    total_words = len(tokens)
    if total_words == 0:
        return [0, 0, 0, 0, 0]  # Если токенов нет, возвращаем нули

    # Возвращаем относительные частоты
    return [
        pos_counts["NOUN"] / total_words,  # Доля существительных
        pos_counts["VERB"] / total_words,  # Доля глаголов
        pos_counts["ADJ"] / total_words,  # Доля прилагательных
        pos_counts["ADV"] / total_words,  # Доля наречий
        pos_counts["OTHER"] / total_words,  # Доля других частей речи
    ]


# Применение функции к токенам
df["morph_features"] = df["tokens"].apply(extract_morph_features)

# Разделение списка признаков на отдельные колонки
df[["noun_freq", "verb_freq", "adj_freq", "adv_freq", "other_freq"]] = pd.DataFrame(
    df["morph_features"].tolist(), index=df.index
)

# Удаление временной колонки с признаками
df.drop(columns=["morph_features"], inplace=True)

# Сохранение результатов в файл
df.to_csv("text_with_morph_features.csv", index=False)

In [16]:
df

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,noun_freq,verb_freq,adj_freq,adv_freq,other_freq
0,на работе был полный пиддес :| и так каждое за...,-1,работа полный пиддес каждый закрытие месяц сви...,"[работа, полный, пиддес, каждый, закрытие, мес...",0.129630,0.222222,0.000000,0.388889,2,0,0.571429,0.0,0.0,0.0,0.428571
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,коллега сидеть рубиться изз долбать винд мочь,"[коллега, сидеть, рубиться, изз, долбать, винд...",-0.200000,-0.200000,-0.200000,-0.200000,0,1,0.428571,0.0,0.0,0.0,0.571429
2,@elina_4post как говорят обещаного три года жд...,-1,говорить обещаной год ждать,"[говорить, обещаной, год, ждать]",0.000000,0.000000,0.000000,0.000000,0,0,0.250000,0.0,0.0,0.0,0.750000
3,"Желаю хорошего полёта и удачной посадки,я буду...",-1,желать хороший полёт удачный посадкия быть оче...,"[желать, хороший, полёт, удачный, посадкия, бы...",1.111111,1.222222,1.000000,2.222222,2,0,0.222222,0.0,0.0,0.0,0.777778
4,"Обновил за каким-то лешим surf, теперь не рабо...",-1,обновить какимтый леший работать простоплеер,"[обновить, какимтый, леший, работать, простопл...",0.000000,0.000000,0.000000,0.000000,0,0,0.400000,0.0,0.0,0.0,0.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[каждый, хотеть, исправлять]",0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0.0,0.0,0.0,1.000000
111919,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать, вправлять, мозг, равно, скучать]",0.000000,0.000000,0.000000,0.000000,0,0,0.200000,0.0,0.0,0.0,0.800000
111920,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[школа, говно, это, идти]",0.000000,0.000000,0.000000,0.000000,0,0,0.500000,0.0,0.0,0.0,0.500000
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[тауриэль, грусть, обнять]",0.000000,0.000000,0.000000,0.000000,0,0,0.666667,0.0,0.0,0.0,0.333333


In [17]:
# Инициализация TfidfVectorizer с ограничением на 1000 признаков
vectorizer = TfidfVectorizer(max_features=1000)

# Преобразование текста в TF-IDF признаки
tfidf_features = vectorizer.fit_transform(df["normalized_text"]).toarray()

# Создание DataFrame с TF-IDF признаками
tfidf_df = pd.DataFrame(
    tfidf_features, columns=[f"tfidf_{i}" for i in range(tfidf_features.shape[1])]
)

# Сбросить индексы в обоих DataFrame
df = df.reset_index(drop=True)
tfidf_df = tfidf_df.reset_index(drop=True)

# Объединение TF-IDF признаков с исходным DataFrame
df = pd.concat([df, tfidf_df], axis=1)

# Сохранение результатов в файл
df.to_csv("text_with_tfidf_features.csv", index=False)

In [ ]:
df

In [18]:
# Обучение модели Word2Vec
word2vec_model = Word2Vec(
    sentences=df["tokens"],  # Токенизированные тексты
    vector_size=300,  # Размерность векторов
    window=5,  # Размер окна (сколько слов вокруг учитывать)
    min_count=1,  # Минимальная частота слова для учета
    workers=4,  # Количество потоков для обучения
)


# Функция для извлечения среднего вектора текста
def get_text_vector(tokens, model):
    # Фильтруем слова, которые есть в модели
    vectors = [model.wv[word] for word in tokens if word in model.wv]

    # Если в тексте нет слов из модели, возвращаем нулевой вектор
    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    # Вычисляем средний вектор
    return np.mean(vectors, axis=0)


# Применение функции к каждому тексту
word2vec_features = np.array(
    [get_text_vector(tokens, word2vec_model) for tokens in df["tokens"]]
)

# Создание DataFrame с Word2Vec-признаками
word2vec_df = pd.DataFrame(
    word2vec_features, columns=[f"w2v_{i}" for i in range(word2vec_features.shape[1])]
)

# Объединение Word2Vec-признаков с исходным DataFrame
df = pd.concat([df, word2vec_df], axis=1)

# Сохранение результатов в файл
df.to_csv("text_with_word2vec_features.csv", index=False)

In [19]:
df

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,...,w2v_290,w2v_291,w2v_292,w2v_293,w2v_294,w2v_295,w2v_296,w2v_297,w2v_298,w2v_299
0,на работе был полный пиддес :| и так каждое за...,-1,работа полный пиддес каждый закрытие месяц сви...,"[работа, полный, пиддес, каждый, закрытие, мес...",0.129630,0.222222,0.000000,0.388889,2,0,...,0.044692,0.527794,0.320332,0.051172,0.234894,0.712189,0.212943,-0.115083,0.146501,0.184508
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,коллега сидеть рубиться изз долбать винд мочь,"[коллега, сидеть, рубиться, изз, долбать, винд...",-0.200000,-0.200000,-0.200000,-0.200000,0,1,...,0.289401,0.726005,0.427902,-0.010553,0.568594,0.535854,-0.030831,-0.146549,0.210963,-0.133485
2,@elina_4post как говорят обещаного три года жд...,-1,говорить обещаной год ждать,"[говорить, обещаной, год, ждать]",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.103194,0.128916,0.217479,0.663824,0.087138,1.324087,0.274727,-0.242443,0.420791,0.062301
3,"Желаю хорошего полёта и удачной посадки,я буду...",-1,желать хороший полёт удачный посадкия быть оче...,"[желать, хороший, полёт, удачный, посадкия, бы...",1.111111,1.222222,1.000000,2.222222,2,0,...,-0.118332,0.629948,0.193159,0.441973,0.482163,0.752155,0.126101,-0.088976,0.127751,-0.041167
4,"Обновил за каким-то лешим surf, теперь не рабо...",-1,обновить какимтый леший работать простоплеер,"[обновить, какимтый, леший, работать, простопл...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.045780,0.384433,0.376364,0.136653,0.277619,0.095153,-0.218836,-0.259370,0.490778,-0.068122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223841,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[каждый, хотеть, исправлять]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.037201,1.076488,0.578004,0.327114,0.319456,0.672960,0.145601,-0.142732,-0.095584,0.147779
223842,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать, вправлять, мозг, равно, скучать]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.000138,0.478831,0.305380,0.484312,0.693100,0.907580,0.156042,-0.002473,-0.050191,-0.009064
223843,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[школа, говно, это, идти]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.053365,0.888563,0.613432,0.294541,0.666554,0.911809,0.171644,-0.189071,0.536712,-0.477648
223844,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[тауриэль, грусть, обнять]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.044750,0.366292,0.157736,0.132367,0.370507,0.263584,0.008126,-0.031942,0.100131,-0.142943


In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# Загрузка данных
df = pd.read_csv("text_with_word2vec_features.csv")

# Разделение данных на признаки (X) и целевую переменную (y)
X = df.drop(
    columns=["text", "type", "normalized_text", "tokens"]
)  # Убираем ненужные колонки
y = df["type"]  # Целевая переменная

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Создание и обучение модели
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Предсказание на тестовой выборке
y_pred = model.predict(X_test)

ValueError: pos_label=1 is not a valid label. It should be one of [-1, 0]

In [21]:
# Вычисление метрик
precision = precision_score(y_test, y_pred, average="macro")
recall = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

# Вывод результатов
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

# Отчет о классификации
report = classification_report(y_test, y_pred)
print(report)

Precision: 0.4876
Recall: 0.4877
F1-score: 0.4863
              precision    recall  f1-score   support

          -1       0.49      0.54      0.51     22312
           0       0.49      0.44      0.46     22458

    accuracy                           0.49     44770
   macro avg       0.49      0.49      0.49     44770
weighted avg       0.49      0.49      0.49     44770



In [22]:
import subprocess
import os


def preprocess_text_with_udpipe(text):
    try:
        # Сохраняем текст во временный файл
        with open("temp_input.txt", "w", encoding="utf-8") as f:
            f.write(text)

        # Вызов скрипта UDPipe
        result = subprocess.run(
            [
                "python3",
                BASE_PATH + "/rus_preprocessing_udpipe.py",
                "--input",
                "temp_input.txt",
                "--output",
                "temp_output.txt",
            ],
            capture_output=True,
            text=True,
        )

        # Проверка, завершился ли скрипт успешно
        if result.returncode != 0:
            raise RuntimeError(f"Скрипт завершился с ошибкой: {result.stderr}")

        # Проверка, создан ли выходной файл
        if not os.path.exists("temp_output.txt"):
            raise FileNotFoundError("Выходной файл не был создан.")

        # Чтение результата
        with open("temp_output.txt", "r", encoding="utf-8") as f:
            processed_text = f.read()

        return processed_text

    except Exception as e:
        print(f"Ошибка при обработке текста: {e}")
        return text

In [24]:
from ufal.udpipe import Model, Pipeline
import pandas as pd

# Загрузка модели UDPipe (один раз)
model_path = BASE_PATH + "/udpipe_syntagrus.model"
model = Model.load(model_path)
if not model:
    raise Exception("Не удалось загрузить модель UDPipe.")
pipeline = Pipeline(model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu")


# Функция для обработки текста с использованием UDPipe
def preprocess_text_with_udpipe(text):
    processed = pipeline.process(text)
    # Извлечение лемм и частей речи (пример из вашего скрипта)
    content = [line for line in processed.split("\n") if not line.startswith("#")]
    tagged = [w.split("\t") for w in content if w]
    lemmas = []
    for t in tagged:
        if len(t) != 10:
            continue
        lemma = t[2]  # Лемма
        pos = t[3]  # Часть речи
        lemmas.append(f"{lemma}_{pos}")
    return " ".join(lemmas)


# Применение функции к каждому тексту в dfudpipe
dfudpipe["processed_text"] = dfudpipe["text"].apply(preprocess_text_with_udpipe)

# Токенизация предобработанного текста
dfudpipe["tokens"] = dfudpipe["processed_text"].apply(lambda x: x.split())

dfudpipe

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,noun_freq,verb_freq,adj_freq,adv_freq,other_freq,processed_text
0,на работе был полный пиддес :| и так каждое за...,-1,работа полный пиддес каждый закрытие месяц сви...,"[на_ADP, работа_NOUN, быть_AUX, полный_ADJ, пи...",0.129630,0.222222,0.000000,0.388889,2,0,0.571429,0.0,0.0,0.0,0.428571,на_ADP работа_NOUN быть_AUX полный_ADJ пиддо_N...
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,коллега сидеть рубиться изз долбать винд мочь,"[коллега_NOUN, сидеть_VERB, рубиться_VERB, в_A...",-0.200000,-0.200000,-0.200000,-0.200000,0,1,0.428571,0.0,0.0,0.0,0.571429,коллега_NOUN сидеть_VERB рубиться_VERB в_ADP U...
2,@elina_4post как говорят обещаного три года жд...,-1,говорить обещаной год ждать,"[@elina_4post_X, как_SCONJ, говорить_VERB, обе...",0.000000,0.000000,0.000000,0.000000,0,0,0.250000,0.0,0.0,0.0,0.750000,@elina_4post_X как_SCONJ говорить_VERB обещаны...
3,"Желаю хорошего полёта и удачной посадки,я буду...",-1,желать хороший полёт удачный посадкия быть оче...,"[желать_VERB, хороший_ADJ, полет_NOUN, и_CCONJ...",1.111111,1.222222,1.000000,2.222222,2,0,0.222222,0.0,0.0,0.0,0.777778,желать_VERB хороший_ADJ полет_NOUN и_CCONJ уда...
4,"Обновил за каким-то лешим surf, теперь не рабо...",-1,обновить какимтый леший работать простоплеер,"[обновить_VERB, за_ADP, какой-то_ADJ, лешимый_...",0.000000,0.000000,0.000000,0.000000,0,0,0.400000,0.0,0.0,0.0,0.600000,обновить_VERB за_ADP какой-то_ADJ лешимый_ADJ ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[но_CCONJ, не_PART, каждый_ADJ, хотеть_VERB, ч...",0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0.0,0.0,0.0,1.000000,но_CCONJ не_PART каждый_ADJ хотеть_VERB что_PR...
111919,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать_VERB, так_ADV, :_PUNCT, -_PUNCT, (_PU...",0.000000,0.000000,0.000000,0.000000,0,0,0.200000,0.0,0.0,0.0,0.800000,скучать_VERB так_ADV :_PUNCT -_PUNCT (_PUNCT т...
111920,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[вот_PART, и_PART, в_ADP, школа_NOUN, ,_PUNCT,...",0.000000,0.000000,0.000000,0.000000,0,0,0.500000,0.0,0.0,0.0,0.500000,"вот_PART и_PART в_ADP школа_NOUN ,_PUNCT в_ADP..."
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[Rt_PROPN, @_Them___NOUN, :_PUNCT, @LisaBeroud...",0.000000,0.000000,0.000000,0.000000,0,0,0.666667,0.0,0.0,0.0,0.333333,Rt_PROPN @_Them___NOUN :_PUNCT @LisaBeroud_PRO...


In [25]:
# Сохранение результата в CSV-файл
output_file = "processed_texts.csv"
dfudpipe.to_csv(output_file, index=False, encoding="utf-8")

print(f"Результат сохранен в файл: {output_file}")

Результат сохранен в файл: processed_texts.csv


In [31]:
word2vec_model_path = BASE_PATH + "/model.bin"
word2vec_model_path

'C:/work projects/ITMO/ZOS/NLP/DATA/model.bin'

In [30]:
from gensim.models import KeyedVectors

# 3. Загрузка предобученной модели Word2Vec
word2vec_model_path = BASE_PATH + "/model.bin"
word2vec_model = KeyedVectors.load_word2vec_format(word2vec_model_path, binary=True)

In [32]:
# Функция для получения среднего вектора текста
def get_text_vector(tokens, model):
    vectors = []
    for token in tokens:
        if token in model:  # Проверяем, есть ли слово в модели
            vectors.append(model[token])
    if len(vectors) == 0:
        return np.zeros(
            model.vector_size
        )  # Возвращаем нулевой вектор, если слов нет в модели
    return np.mean(vectors, axis=0)  # Усредняем векторы


# Применение функции к предобработанным текстам
dfudpipe["word2vec_vector"] = dfudpipe["tokens"].apply(
    lambda x: get_text_vector(x, word2vec_model)
)

# 4. Преобразование векторов в отдельные столбцы
word2vec_columns = pd.DataFrame(
    dfudpipe["word2vec_vector"].tolist(),
    columns=[f"w2v_{i}" for i in range(word2vec_model.vector_size)],
)

# Сброс индекса для dfudpipe и word2vec_columns
dfudpipe = dfudpipe.reset_index(drop=True)
word2vec_columns = word2vec_columns.reset_index(drop=True)

# Объединение с исходным DataFrame
dfudpipe = pd.concat([dfudpipe, word2vec_columns], axis=1)

# 5. Сохранение результата в CSV
output_file = "texts_with_word2vec_vectors.csv"
dfudpipe.to_csv(output_file, index=False, encoding="utf-8")

print(f"Результат сохранен в файл: {output_file}")

dfudpipe

Результат сохранен в файл: texts_with_word2vec_vectors.csv


,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,...,w2v_290,w2v_291,w2v_292,w2v_293,w2v_294,w2v_295,w2v_296,w2v_297,w2v_298,w2v_299
0,на работе был полный пиддес :| и так каждое за...,-1,работа полный пиддес каждый закрытие месяц сви...,"[на_ADP, работа_NOUN, быть_AUX, полный_ADJ, пи...",0.129630,0.222222,0.000000,0.388889,2,0,...,-1.497247,-0.617494,0.155409,-0.502912,0.546641,0.577736,1.327193,0.337428,0.970455,2.135193
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,коллега сидеть рубиться изз долбать винд мочь,"[коллега_NOUN, сидеть_VERB, рубиться_VERB, в_A...",-0.200000,-0.200000,-0.200000,-0.200000,0,1,...,-0.769979,0.683749,1.193472,-1.152344,0.130662,-0.131944,0.294939,-0.399048,-0.114051,-0.866398
2,@elina_4post как говорят обещаного три года жд...,-1,говорить обещаной год ждать,"[@elina_4post_X, как_SCONJ, говорить_VERB, обе...",0.000000,0.000000,0.000000,0.000000,0,0,...,1.204233,-0.250001,1.255472,-1.004997,1.619486,-0.016101,-1.633986,0.852788,0.842799,1.491378
3,"Желаю хорошего полёта и удачной посадки,я буду...",-1,желать хороший полёт удачный посадкия быть оче...,"[желать_VERB, хороший_ADJ, полет_NOUN, и_CCONJ...",1.111111,1.222222,1.000000,2.222222,2,0,...,0.072630,0.976666,-0.673876,0.083581,-0.047168,0.259534,1.448931,-0.508064,0.257483,0.560332
4,"Обновил за каким-то лешим surf, теперь не рабо...",-1,обновить какимтый леший работать простоплеер,"[обновить_VERB, за_ADP, какой-то_ADJ, лешимый_...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.054853,-0.174235,-0.065226,0.472297,-0.888250,0.440584,-0.925492,1.001698,-0.069915,0.528838
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223841,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[но_CCONJ, не_PART, каждый_ADJ, хотеть_VERB, ч...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.512176,0.357525,0.800973,0.039046,-0.067466,0.150615,0.447536,-0.007745,-0.992812,-0.197276
223842,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать_VERB, так_ADV, :_PUNCT, -_PUNCT, (_PU...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.341404,-0.602341,-0.188585,-0.132261,-0.473496,-0.570752,0.723195,0.043739,-0.574560,0.907792
223843,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[вот_PART, и_PART, в_ADP, школа_NOUN, ,_PUNCT,...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.077833,0.146524,-0.550453,0.847372,-0.116148,-1.111711,-0.078653,1.274752,0.777959,-1.164700
223844,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[Rt_PROPN, @_Them___NOUN, :_PUNCT, @LisaBeroud...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [36]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# 1. Подготовка данных
# X - признаки (Word2Vec-векторы), y - целевая переменная (type)
X = dfudpipe[[f"w2v_{i}" for i in range(word2vec_model.vector_size)]]
y = dfudpipe["type"]

# 2. Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Обучение модели (например, Random Forest)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 4. Предсказание на тестовой выборке
y_pred = model.predict(X_test)

# 5. Оценка модели
# Вывод отчета о классификации
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

          -1       0.13      0.14      0.14     22312
           0       0.12      0.12      0.12     22458

    accuracy                           0.13     44770
   macro avg       0.13      0.13      0.13     44770
weighted avg       0.13      0.13      0.13     44770



In [35]:
# Отдельные метрики
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro")
recall = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Accuracy: 0.1278
Precision: 0.1276
Recall: 0.1278
F1-Score: 0.1277


In [37]:
from gensim.models import FastText
import numpy as np

# 1. Обучение FastText-модели на ваших данных
fasttext_model = FastText(
    sentences=dfudpipe["tokens"], vector_size=100, window=5, min_count=1, workers=4
)


# 2. Функция для усреднения векторов слов в тексте
def get_fasttext_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(
            model.vector_size
        )  # Возвращаем нулевой вектор, если слов нет в модели
    return np.mean(vectors, axis=0)  # Усредняем векторы


# 3. Применение функции к каждому тексту
dfudpipe["fasttext_vector"] = dfudpipe["tokens"].apply(
    lambda x: get_fasttext_vector(x, fasttext_model)
)

# 4. Преобразование векторов в отдельные столбцы
fasttext_columns = pd.DataFrame(
    dfudpipe["fasttext_vector"].tolist(),
    columns=[f"ft_{i}" for i in range(fasttext_model.vector_size)],
)

# 5. Объединение с исходным DataFrame
dfudpipe = pd.concat([dfudpipe, fasttext_columns], axis=1)

# 6. Сохранение результата
dfudpipe.to_csv("texts_with_fasttext_vectors.csv", index=False, encoding="utf-8")
print("FastText-векторы сохранены в texts_with_fasttext_vectors.csv")

dfudpipe

FastText-векторы сохранены в texts_with_fasttext_vectors.csv


,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,...,ft_90,ft_91,ft_92,ft_93,ft_94,ft_95,ft_96,ft_97,ft_98,ft_99
0,на работе был полный пиддес :| и так каждое за...,-1,работа полный пиддес каждый закрытие месяц сви...,"[на_ADP, работа_NOUN, быть_AUX, полный_ADJ, пи...",0.129630,0.222222,0.000000,0.388889,2,0,...,-0.775222,-1.874399,0.658910,-0.494101,1.087634,-0.927668,-1.216484,-0.468666,-1.692155,0.405708
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,коллега сидеть рубиться изз долбать винд мочь,"[коллега_NOUN, сидеть_VERB, рубиться_VERB, в_A...",-0.200000,-0.200000,-0.200000,-0.200000,0,1,...,-0.371942,-1.630330,0.603449,-0.755360,1.554826,-1.036633,-0.999218,0.501000,-1.728602,0.385504
2,@elina_4post как говорят обещаного три года жд...,-1,говорить обещаной год ждать,"[@elina_4post_X, как_SCONJ, говорить_VERB, обе...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.784104,-1.534507,0.822968,-0.258389,0.766144,-1.622127,-1.129928,-0.844899,-1.856274,0.256184
3,"Желаю хорошего полёта и удачной посадки,я буду...",-1,желать хороший полёт удачный посадкия быть оче...,"[желать_VERB, хороший_ADJ, полет_NOUN, и_CCONJ...",1.111111,1.222222,1.000000,2.222222,2,0,...,-0.850958,-1.051339,1.380642,0.050276,1.557833,-1.340715,-1.333217,0.098030,-1.639469,0.101360
4,"Обновил за каким-то лешим surf, теперь не рабо...",-1,обновить какимтый леший работать простоплеер,"[обновить_VERB, за_ADP, какой-то_ADJ, лешимый_...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.403881,-1.104045,0.288947,-0.522843,1.012826,-1.605133,-0.746090,-0.646005,-1.532612,0.246512
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223841,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[но_CCONJ, не_PART, каждый_ADJ, хотеть_VERB, ч...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.177440,-2.405580,1.830889,-0.087753,0.377675,-1.509996,-0.361875,0.564261,-2.888892,-0.181214
223842,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать_VERB, так_ADV, :_PUNCT, -_PUNCT, (_PU...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.651255,-1.324231,1.436762,0.026069,1.618569,-0.402255,-0.469552,-0.057970,-1.849186,0.101151
223843,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[вот_PART, и_PART, в_ADP, школа_NOUN, ,_PUNCT,...",0.000000,0.000000,0.000000,0.000000,0,0,...,-1.161098,-2.451230,0.587367,-0.458864,-0.573189,-1.894255,-1.275983,-0.750601,-2.300856,-0.165546
223844,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[Rt_PROPN, @_Them___NOUN, :_PUNCT, @LisaBeroud...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.306920,-0.451451,0.356278,-0.257237,1.471719,0.415706,-1.300882,-0.191673,-1.406489,1.231041


In [39]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# 1. Подготовка данных
# X - FastText-векторы, y - целевая переменная (type)
X = dfudpipe[[f"ft_{i}" for i in range(fasttext_model.vector_size)]]
y = dfudpipe["type"]

# 2. Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Обучение модели (например, Random Forest)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 4. Предсказание на тестовой выборке
y_pred = model.predict(X_test)

# 5. Оценка модели
# Вывод отчета о классификации
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

          -1       0.11      0.11      0.11     22312
           0       0.11      0.11      0.11     22458

    accuracy                           0.11     44770
   macro avg       0.11      0.11      0.11     44770
weighted avg       0.11      0.11      0.11     44770



In [40]:
# Отдельные метрики
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro")
recall = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Accuracy: 0.1071
Precision: 0.1071
Recall: 0.1071
F1-Score: 0.1071


In [42]:
import numpy as np
import pandas as pd
from gensim.models import KeyedVectors

# Загрузка данных из processed_texts.csv
processed_texts_path = "processed_texts.csv"  # Укажите путь к файлу
dfft = pd.read_csv(processed_texts_path)

# Проверка структуры данных
print(dfft.head())  # Убедимся, что данные загружены правильно

                                                text  type  \
0  на работе был полный пиддес :| и так каждое за...    -1   
1  Коллеги сидят рубятся в Urban terror, а я из-з...    -1   
2  @elina_4post как говорят обещаного три года жд...    -1   
3  Желаю хорошего полёта и удачной посадки,я буду...    -1   
4  Обновил за каким-то лешим surf, теперь не рабо...    -1   

                                     normalized_text  \
0  работа полный пиддес каждый закрытие месяц сви...   
1      коллега сидеть рубиться изз долбать винд мочь   
2                        говорить обещаной год ждать   
3  желать хороший полёт удачный посадкия быть оче...   
4       обновить какимтый леший работать простоплеер   

                                              tokens  tonality_mean  \
0  ['на_ADP', 'работа_NOUN', 'быть_AUX', 'полный_...       0.129630   
1  ['коллега_NOUN', 'сидеть_VERB', 'рубиться_VERB...      -0.200000   
2  ['@elina_4post_X', 'как_SCONJ', 'говорить_VERB...       0.000000   
3  ['ж

In [46]:
from gensim.models import KeyedVectors

fasttext_model_path = BASE_PATH + "/cc.ru.300.vec"
fasttext_model = KeyedVectors.load_word2vec_format(
    fasttext_model_path, binary=False, limit=500_000
)

In [47]:
# Функция для извлечения среднего вектора текста с использованием предобученной модели FastText
def get_fasttext_vector(tokens, model):
    # Фильтруем слова, которые есть в модели
    vectors = [model[word] for word in tokens if word in model]

    # Если в тексте нет слов из модели, возвращаем нулевой вектор
    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    # Вычисляем средний вектор
    return np.mean(vectors, axis=0)


# Предположим, что в dfft есть столбец 'tokens' с токенизированными текстами
# Если токены хранятся в виде строки, преобразуем их в список
if isinstance(dfft["tokens"].iloc[0], str):
    dfft["tokens"] = dfft["tokens"].apply(lambda x: x.split())

# Применение функции к каждому тексту
fasttext_features = np.array(
    [get_fasttext_vector(tokens, fasttext_model) for tokens in dfft["tokens"]]
)

# Создание DataFrame с FastText-признаками
fasttext_df = pd.DataFrame(
    fasttext_features,
    columns=[f"fasttext_{i}" for i in range(fasttext_features.shape[1])],
)

# Объединение FastText-признаков с исходным DataFrame
dfft = pd.concat([dfft, fasttext_df], axis=1)

# Сохранение результатов в файл
dfft.to_csv("text_with_fasttext_features.csv", index=False)

# Вывод результата
print(dfft.head())

                                                text  type  \
0  на работе был полный пиддес :| и так каждое за...    -1   
1  Коллеги сидят рубятся в Urban terror, а я из-з...    -1   
2  @elina_4post как говорят обещаного три года жд...    -1   
3  Желаю хорошего полёта и удачной посадки,я буду...    -1   
4  Обновил за каким-то лешим surf, теперь не рабо...    -1   

                                     normalized_text  \
0  работа полный пиддес каждый закрытие месяц сви...   
1      коллега сидеть рубиться изз долбать винд мочь   
2                        говорить обещаной год ждать   
3  желать хороший полёт удачный посадкия быть оче...   
4       обновить какимтый леший работать простоплеер   

                                              tokens  tonality_mean  \
0  [['на_ADP',, 'работа_NOUN',, 'быть_AUX',, 'пол...       0.129630   
1  [['коллега_NOUN',, 'сидеть_VERB',, 'рубиться_V...      -0.200000   
2  [['@elina_4post_X',, 'как_SCONJ',, 'говорить_V...       0.000000   
3  [['

In [48]:
dfft

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,...,fasttext_290,fasttext_291,fasttext_292,fasttext_293,fasttext_294,fasttext_295,fasttext_296,fasttext_297,fasttext_298,fasttext_299
0,на работе был полный пиддес :| и так каждое за...,-1,работа полный пиддес каждый закрытие месяц сви...,"[['на_ADP',, 'работа_NOUN',, 'быть_AUX',, 'пол...",0.129630,0.222222,0.000000,0.388889,2,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,коллега сидеть рубиться изз долбать винд мочь,"[['коллега_NOUN',, 'сидеть_VERB',, 'рубиться_V...",-0.200000,-0.200000,-0.200000,-0.200000,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,@elina_4post как говорят обещаного три года жд...,-1,говорить обещаной год ждать,"[['@elina_4post_X',, 'как_SCONJ',, 'говорить_V...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,"Желаю хорошего полёта и удачной посадки,я буду...",-1,желать хороший полёт удачный посадкия быть оче...,"[['желать_VERB',, 'хороший_ADJ',, 'полет_NOUN'...",1.111111,1.222222,1.000000,2.222222,2,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,"Обновил за каким-то лешим surf, теперь не рабо...",-1,обновить какимтый леший работать простоплеер,"[['обновить_VERB',, 'за_ADP',, 'какой-то_ADJ',...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223841,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[['но_CCONJ',, 'не_PART',, 'каждый_ADJ',, 'хот...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223842,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[['скучать_VERB',, 'так_ADV',, ':_PUNCT',, '-_...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223843,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[['вот_PART',, 'и_PART',, 'в_ADP',, 'школа_NOU...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223844,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[['Rt_PROPN',, '@_Them___NOUN',, ':_PUNCT',, '...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [50]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# Убедимся, что в X только числовые признаки
# Удаляем все нечисловые столбцы, такие как 'text', 'tokens', 'normalized_text'
X = dfft.drop(
    columns=["text", "tokens", "normalized_text", "type", "processed_text"]
)  # Удаляем текстовые столбцы
y = dfft["type"]  # Целевая переменная

# Проверка типов данных в X
print("Типы данных в X:")
print(X.dtypes)

# Проверка первых строк X
print("\nПервые строки X:")
print(X.head())

# Проверка на пропущенные значения
print("\nПропущенные значения в X:")
print(X.isnull().sum())

# Разделим данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Инициализация и обучение модели (например, Logistic Regression)
model = LogisticRegression(max_iter=1000)  # Увеличиваем max_iter для сходимости
model.fit(X_train, y_train)

# Предсказания на тестовой выборке
y_pred = model.predict(X_test)

Типы данных в X:
tonality_mean         float64
tonality_max          float64
tonality_min          float64
tonality_sum          float64
tonality_pos_count      int64
                       ...   
fasttext_295          float64
fasttext_296          float64
fasttext_297          float64
fasttext_298          float64
fasttext_299          float64
Length: 311, dtype: object

Первые строки X:
   tonality_mean  tonality_max  tonality_min  tonality_sum  \
0       0.129630      0.222222           0.0      0.388889   
1      -0.200000     -0.200000          -0.2     -0.200000   
2       0.000000      0.000000           0.0      0.000000   
3       1.111111      1.222222           1.0      2.222222   
4       0.000000      0.000000           0.0      0.000000   

   tonality_pos_count  tonality_neg_count  noun_freq  verb_freq  adj_freq  \
0                   2                   0   0.571429        0.0       0.0   
1                   0                   1   0.428571        0.0       0.0   
2   

In [51]:
# Оценка метрик
precision = precision_score(
    y_test, y_pred, average="macro"
)  # Для бинарной классификации
recall = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

# Вывод метрик
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

# Детальный отчёт по метрикам
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Class 0", "Class 1"]))

Precision: 0.5020
Recall: 0.5014
F1-score: 0.4564

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.50      0.79      0.61     22312
     Class 1       0.50      0.22      0.30     22458

    accuracy                           0.50     44770
   macro avg       0.50      0.50      0.46     44770
weighted avg       0.50      0.50      0.46     44770



# F1 
word2vec мой     0.128

word2vec чужой   0.486 

fasttext мой     0.107

fasttext чужой   0.456
